## Preprocesamiento de los datos

In [1]:
# Cargamos los paquetes necesarios ACORDARNOS DE HACER EL AMBIENTE VIRTUAL
import polars as pl
import numpy as np
import pyprojroot
import pandas as pd
from sklearn.impute import SimpleImputer

In [2]:
# Definir la ruta raiz del proyecto
ROOT = pyprojroot.here()

# Cargamos los datos
datos_entrenamiento = pl.read_parquet(ROOT / "datos" / "temporada1.parquet")
datos_validacion = pl.read_parquet(ROOT / "datos" / "temporada2.parquet")

In [3]:
# Identificamos que variables tienen valores nulos y la cantidad en cada una
na = (
    datos_entrenamiento
    .null_count()
    .transpose(include_header=True)
    .filter(pl.col("column_0") > 0)
)

na

column,column_0
str,u32
"""release_speed""",367
"""pitch_type""",367
"""pfx_x""",3033
"""pfx_z""",1060
"""plate_x""",367
"""plate_z""",400
"""sz_top""",367
"""sz_bot""",412
"""altura_zona""",412


In [4]:
# Se seleccionan las observaciones que contienen al menos un valor nulo
datos_con_na = datos_entrenamiento.filter(
    pl.any_horizontal(pl.all().is_null())
)

datos_con_na.height

3802

In [5]:
# Se cuenta la cantidad de valores nulos por fila y resumimos cuántas observaciones tienen
datos_entrenamiento.with_columns(
    pl.sum_horizontal(pl.all().is_null()).alias("NA_por_fila")
).filter(
    pl.col("NA_por_fila") > 0
).group_by("NA_por_fila").len().sort("NA_por_fila")

NA_por_fila,len
u32,u32
1,3388
2,47
9,367


In [6]:
# Se muestra las observaciones que presentan 9 o más valores nulos
datos_entrenamiento.with_columns(
    pl.sum_horizontal(pl.all().is_null()).alias("NA_por_fila")
).filter(
    pl.col("NA_por_fila") >= 9
)

release_speed,description,stand,p_throws,pitch_type,balls,strikes,pfx_x,pfx_z,plate_x,plate_z,sz_top,sz_bot,altura_zona,swing,NA_por_fila
f64,str,str,str,str,i64,i64,f64,f64,f64,f64,f64,f64,f64,i32,u32
null,"""ball""","""R""","""R""",null,3,0,null,null,null,null,null,null,null,0,9
null,"""ball""","""L""","""R""",null,1,0,null,null,null,null,null,null,null,0,9
null,"""called_strike""","""R""","""L""",null,1,0,null,null,null,null,null,null,null,0,9
null,"""ball""","""R""","""L""",null,0,1,null,null,null,null,null,null,null,0,9
null,"""called_strike""","""R""","""L""",null,0,1,null,null,null,null,null,null,null,0,9
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
null,"""swinging_strike""","""R""","""L""",null,0,0,null,null,null,null,null,null,null,1,9
null,"""ball""","""L""","""R""",null,1,1,null,null,null,null,null,null,null,0,9
null,"""foul""","""R""","""R""",null,3,2,null,null,null,null,null,null,null,1,9



Se realizó un análisis de los valores faltantes presentes en el conjunto de entrenamiento. Se identificaron 3.802 observaciones con al menos un valor ausente sobre un total de 709.852 registros, lo que representa aproximadamente un 0,54% de los datos. Los faltantes se concentran principalmente en variables relacionadas con las características físicas y espaciales de los lanzamientos, tales como la velocidad (`release_speed`), el tipo de lanzamiento (`pitch_type`), el movimiento de la pelota (`pfx_x` y `pfx_z`) y la ubicación del lanzamiento respecto de la zona de strike (`plate_x`, `plate_z`, `sz_top`, `sz_bot` y `altura_zona`).

Si bien la proporción de observaciones incompletas es reducida, la consigna del trabajo requiere generar una predicción para cada lanzamiento presente en `temporada2.parquet`. En consecuencia, la eliminación de registros con valores faltantes podría ocasionar una pérdida de observaciones durante la etapa de validación y dificultar la generación de predicciones para la totalidad de los lanzamientos.

Por este motivo, se opta por imputar los valores faltantes. Para las variables numéricas se utiliza la mediana calculada sobre los datos de entrenamiento. En el caso de las variables categóricas, los valores ausentes se reemplazan por la categoría más frecuente observada en el conjunto de entrenamiento, es decir, la moda.

### Imputación de valores faltantes



Luego del análisis realizado, se aplica la estrategia de imputación seleccionada sobre los conjuntos de entrenamiento y validación. Para evitar la incorporación de información proveniente del conjunto de validación, los parámetros de imputación se calculan únicamente utilizando los datos correspondientes a `temporada1.parquet` y posteriormente se aplican sobre ambos conjuntos.

In [7]:
variables_cuantitativas = [
    "release_speed",
    "balls",
    "strikes",
    "pfx_x",
    "pfx_z",
    "plate_x",
    "plate_z",
    "sz_top",
    "sz_bot",
    "altura_zona"
]

variables_categoricas = [
    "pitch_type",
    "stand",
    "p_throws"
]

In [8]:
# Imputación de variables cuantitativas
imputador_num = SimpleImputer(strategy="median")

datos_entrenamiento[variables_cuantitativas] = imputador_num.fit_transform(
    datos_entrenamiento[variables_cuantitativas]
)

datos_validacion[variables_cuantitativas] = imputador_num.transform(
    datos_validacion[variables_cuantitativas]
)

# Imputación de variables categóricas
# Para que funcione imputador_cat se convirtió el dataframe de polars a uno de pandas
datos_entrenamiento = datos_entrenamiento.to_pandas()
datos_validacion = datos_validacion.to_pandas()

imputador_cat = SimpleImputer(strategy="most_frequent")

# Se calcula la mode del conjunto de entrenamiento y se reemplaza
datos_entrenamiento[variables_categoricas] = imputador_cat.fit_transform(
    datos_entrenamiento[variables_categoricas]
)

# Se aplican las categorias anteriores en el conjunto de validación
datos_validacion[variables_categoricas] = imputador_cat.transform(
    datos_validacion[variables_categoricas]
)

Una vez realizada la imputación, se chequea que ya no haya valores nulos presentes en los dataframes de entrenamiento y validación.

In [9]:
# Confirmación de la no existencia de valores nulos en las variables categoricas
datos_entrenamiento[variables_categoricas].isna().sum()

pitch_type    0
stand         0
p_throws      0
dtype: int64

In [10]:
datos_validacion[variables_categoricas].isna().sum()

pitch_type    0
stand         0
p_throws      0
dtype: int64

In [11]:
# Confirmación de la no existencia de valores nulos en las variables cuantitativas
datos_entrenamiento[variables_cuantitativas].isna().sum()

release_speed    0
balls            0
strikes          0
pfx_x            0
pfx_z            0
plate_x          0
plate_z          0
sz_top           0
sz_bot           0
altura_zona      0
dtype: int64

In [12]:
datos_validacion[variables_cuantitativas].isna().sum()

release_speed    0
balls            0
strikes          0
pfx_x            0
pfx_z            0
plate_x          0
plate_z          0
sz_top           0
sz_bot           0
altura_zona      0
dtype: int64

In [13]:
# Volvemos el DataFrame a polars
datos_entrenamiento = pl.from_pandas(datos_entrenamiento)
datos_validacion = pl.from_pandas(datos_validacion)

In [14]:
# Guardamos el conjunto de entrenamiento imputado
datos_entrenamiento.write_parquet(
    "../datos/temporada1_imputada.parquet"
)

# Guardamos el conjunto de validación imputado
datos_validacion.write_parquet(
    "../datos/temporada2_imputada.parquet"
)


### Tratamiento de valores atípicos (Outliers)

Luego de realizar las imputaciones correspondientes, se analizan los valores atípicos de las variables presentes en el conjunto de datos. Para ello, se utilizará como referencia el análisis descriptivo realizado previamente, el cual permitió comprender las características de las variables. A partir de esta información, se evaluará si los valores atípicos identificados requieren algún tratamiento.

Para continuar, se comparó el análisis descriptivo previo con los límites reales del béisbol. Esto permitió detectar dos tipos de anomalías:
- **Errores reglamentarios**: Un caso aislado donde la variable `balls` tomó el valor de 4 (lo máximo permitido en juego es 3).
- **Lanzamientos extremos**: Valores atípicos en las variables continuas, como la velocidad del lanzamiento (`release_speed`) de 30 mph (lo que indica lanzamientos muy lentos), posición vertical de la pelota (`plate_z`) negativas de -5.07 pies (lo que significa que la pelota picó en el suelo) y posición horizontal de la pelota (`plate_x`) de más de 9 pies (lo que indica que las pelotas se fueron totalmente hacia los costados).

In [15]:
# Se define las condiciones de lo que consideramos "mal" 
condicion_error = (
    (pl.col("balls") > 3) |                       
    (pl.col("release_speed") < 60) |            
    (pl.col("plate_z") < -2) | (pl.col("plate_z") > 7) |  
    (pl.col("plate_x") < -4) | (pl.col("plate_x") > 4)   
)

A partir de estas condiciones, se creó un conjunto de datos denominado `datos_sospechosos`. Este subconjunto agrupa todas las observaciones que presentaron alguna de las condiciones antes mencionadas.

El objetivo es cuantificar cuántos datos representan respecto al total del conjunto de entrenamiento. De esta manera, poder evaluar su impacto porcentual y tomar una decisión fundamentada sobre si corresponde eliminarlos o aplicar otro tipo de tratamiento.

In [16]:
# Se crea el dataset de observaciones sospechosas
datos_sospechosos = datos_entrenamiento.filter(condicion_error)

# Se calculan los números para el análisis
total_filas = datos_entrenamiento.height
cantidad_malos = datos_sospechosos.height
porcentaje_malos = (cantidad_malos / total_filas) * 100

print(f"Total de registros en el dataset: {total_filas}")
print(f"Cantidad de filas con outliers: {cantidad_malos}")
print(f"Porcentaje que representan: {porcentaje_malos:.3f}%")

Total de registros en el dataset: 709852
Cantidad de filas con outliers: 422
Porcentaje que representan: 0.059%


El anterior análisis arrojó un total de 422 observaciones sospechosas sobre los 709.852 registros. Esto significa que los valores atípicos representan apenas el $0.059\%$ del total del conjunto de entrenamiento.

Esta proporción refleja que, si bien existen anomalías físicas y reglamentarias en la captura de los lanzamientos, la cantidad es tan chica que demuestra que estos errores son casos muy aislados y no afectan al resto de los datos.

A partir de los resultados obtenidos, se decidió conservar estas 422 observaciones dentro del conjunto de entrenamiento en lugar de procede. Al representar una cantidad muy pequeña, mantener estos registros permite preservar la integridad del conjunto de datos original sin introducir sesgos ni distorsionar el comportamiento de los futuros modelos predictivos. Asimismo, esta decisión permite reflejar la realidad de los datos, donde conviven jugadas reales y extremas junto con pequeños errores aislados en los sistemas de medición de los estadios.